In [29]:
%pip install pandas


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [30]:
import pandas as pd

columns = ["id", "name", "type", "program", "title",
           "call_sign", "vessel_type", "tonnage", "grt",
           "vessel_flag", "vessel_owner", "remarks"]

sdn = pd.read_csv("../data/raw/sdn.csv", header=None,
                  names=columns, na_values=["-0-", "-0- "])
sdn["id"] = pd.to_numeric(sdn["id"], errors="coerce")
sdn = sdn.dropna(subset=["id"])
sdn["id"] = sdn["id"].astype(int)
sdn.head()

,id,name,type,program,title,call_sign,vessel_type,tonnage,grt,vessel_flag,vessel_owner,remarks
0,36,AEROCARIBBEAN AIRLINES,NaN,CUBA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,173,"ANGLO-CARIBBEAN CO., LTD.",NaN,CUBA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,306,BANCO NACIONAL DE CUBA,NaN,CUBA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,a.k.a. 'BNC'.
3,424,BOUTIQUE LA MAISON,NaN,CUBA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,475,CASA DE CUBA,NaN,CUBA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
len(sdn)

19599

In [32]:
sdn["type"].value_counts(dropna=False)

type
NaN           10122
individual     7565
vessel         1570
aircraft        342
Name: count, dtype: int64

In [33]:
people = sdn[sdn["type"] == "individual"]
people[["name", "program", "remarks"]].head(10)

,name,program,remarks
53,"ABBAS, Abu",SDGT,DOB 10 Dec 1948; Secondary sanctions risk: sec...
54,"AL RAHMAN, Shaykh Umar Abd",SDGT,DOB 03 May 1938; POB Egypt; Secondary sanction...
55,"AL ZAWAHIRI, Dr. Ayman",SDGT,"DOB 19 Jun 1951; POB Giza, Egypt; Secondary sa..."
56,"AL-ZOMOR, Abboud Abdul Latif Hassan",SDGT,"DOB 19 Apr 1947; POB Nahia, Giza, Egypt; natio..."
57,"AWDA, Abd Al Aziz",SDGT,"DOB 20 Dec 1950; POB Jabaliyah, Gaza Strip; na..."
58,"FADLALLAH, Shaykh Muhammad Husayn",SDGT,DOB 1938; alt. DOB 1936; POB Najf Al Ashraf (N...
59,"HAWATMA, Nayif",SDGT,DOB 1933; Secondary sanctions risk: section 1(...
60,"ISLAMBOULI, Mohammad Shawqi",SDGT,DOB 15 Jan 1955; alt. DOB 21 Jan 1952; POB Egy...
61,"JABRIL, Ahmad",SDGT,"DOB 1938; POB Ramleh, Israel; Secondary sancti..."
62,"NAJI, Talal Muhammad Rashid",SDGT,"DOB 1930; POB Al Nasiria, Palestine; Secondary..."


In [34]:
people = people.copy()
people["dob"] = people["remarks"].str.extract(r"DOB ([^;]+)")
people[["name", "dob"]].head(10)

,name,dob
53,"ABBAS, Abu",10 Dec 1948
54,"AL RAHMAN, Shaykh Umar Abd",03 May 1938
55,"AL ZAWAHIRI, Dr. Ayman",19 Jun 1951
56,"AL-ZOMOR, Abboud Abdul Latif Hassan",19 Apr 1947
57,"AWDA, Abd Al Aziz",20 Dec 1950
58,"FADLALLAH, Shaykh Muhammad Husayn",1938
59,"HAWATMA, Nayif",1933
60,"ISLAMBOULI, Mohammad Shawqi",15 Jan 1955
61,"JABRIL, Ahmad",1938
62,"NAJI, Talal Muhammad Rashid",1930


In [35]:
people["nationality"] = people["remarks"].str.extract(r"nationality ([^;]+)")
people["passport"] = people["remarks"].str.extract(r"Passport ([^;]+)")
people[["name", "dob", "nationality", "passport"]].head(10)

,name,dob,nationality,passport
53,"ABBAS, Abu",10 Dec 1948,NaN,NaN
54,"AL RAHMAN, Shaykh Umar Abd",03 May 1938,NaN,NaN
55,"AL ZAWAHIRI, Dr. Ayman",19 Jun 1951,NaN,1084010 (Egypt)
56,"AL-ZOMOR, Abboud Abdul Latif Hassan",19 Apr 1947,Egypt,NaN
57,"AWDA, Abd Al Aziz",20 Dec 1950,Palestinian,NaN
58,"FADLALLAH, Shaykh Muhammad Husayn",1938,NaN,NaN
59,"HAWATMA, Nayif",1933,NaN,NaN
60,"ISLAMBOULI, Mohammad Shawqi",15 Jan 1955,Egypt,304555 (Egypt)
61,"JABRIL, Ahmad",1938,NaN,NaN
62,"NAJI, Talal Muhammad Rashid",1930,NaN,NaN


In [36]:
people[["dob", "nationality", "passport"]].notna().mean()

dob            0.990747
nationality    0.757700
passport       0.278916
dtype: float64

In [37]:
alt = pd.read_csv("../data/raw/alt.csv", header=None,
                  names=["id", "alt_id", "alt_type", "alt_name", "alt_remarks"],
                  na_values=["-0-", "-0- "])
alt["id"] = pd.to_numeric(alt["id"], errors="coerce")
alt = alt.dropna(subset=["id"])
alt["id"] = alt["id"].astype(int)
alt.head()

,id,alt_id,alt_type,alt_name,alt_remarks
0,36,12.0,aka,AERO-CARIBBEAN,NaN
1,173,57.0,aka,AVIA IMPORT,NaN
2,306,220.0,aka,NATIONAL BANK OF CUBA,NaN
3,540,471.0,aka,COIBA,NaN
4,552,475.0,aka,CRYMSA,NaN


In [38]:
alt["alt_type"].value_counts()

alt_type
aka    19733
fka      501
nka       17
Name: count, dtype: int64

In [39]:
people_alt = alt[alt["id"].isin(people["id"])]
len(people_alt)

8656

In [40]:
people_alt["id"].value_counts().head(5)

id
10612    45
21390    25
9340     21
9762     19
6861     18
Name: count, dtype: int64

In [41]:
sdn[sdn["id"] == 10612][["name", "program"]]

,name,program
1100,"DROUKDEL, Abdelmalek",SDGT


In [42]:
people_alt[people_alt["id"] == 10612][["alt_type", "alt_name"]]

,alt_type,alt_name
2241,aka,"DROUKDAL, Abdelmalek"
2242,aka,"DROUKADAL, Abdelmalek"
2243,aka,"DARDAKIL, Abdelmalek"
2244,aka,"DROKDAL, Abdelmalek"
2245,aka,"DOURKDAL, Abdelmalek"
2246,aka,"DROUGDEL, Abdelmalek"
2247,aka,"DROUKBEL, Abdelmalek"
2248,aka,"DERDOUKAL, Abdelmalek"
2249,aka,"DEROUDEL, Abdel Malek"
2250,aka,"DROUKDEL, Abdel Malek"


In [43]:
sdn["id"].dtype, alt["id"].dtype

(dtype('int64'), dtype('int64'))

In [44]:
%pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.3 MB/s  0:00:00

[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [46]:
from rapidfuzz import fuzz

print(fuzz.ratio("DROUKDEL", "DROUKDAL"))
print(fuzz.ratio("Musab Abdel Wadoud", "ABDEL WADOUD, Abou Mossab"))
print(fuzz.token_sort_ratio("Musab Abdel Wadoud", "ABDEL WADOUD, Abou Mossab"))

87.5
23.25581395348837
41.860465116279066


In [47]:
from rapidfuzz import utils

a = "Musab Abdel Wadoud"
b = "ABDEL WADOUD, Abou Mossab"

print(utils.default_process(a))
print(utils.default_process(b))
print(fuzz.token_sort_ratio(a, b, processor=utils.default_process))

musab abdel wadoud
abdel wadoud  abou mossab
80.95238095238095


In [48]:
main_names = people[["id", "name"]]
alias_names = people_alt[["id", "alt_name"]].rename(columns={"alt_name": "name"})

names = pd.concat([main_names, alias_names]).reset_index(drop=True)
len(names)

16221

In [49]:
from rapidfuzz import process

def search(query, limit=5):
    results = process.extract(query, names["name"],
                              scorer=fuzz.token_sort_ratio,
                              processor=utils.default_process,
                              limit=limit)
    rows = []
    for matched_name, score, row_number in results:
        rows.append({"id": names.loc[row_number, "id"],
                     "matched_name": matched_name,
                     "score": round(score)})
    return pd.DataFrame(rows)

search("Musab Abdel Wadoud")

,id,matched_name,score
0,10612,"ABDEL WADOUD, Abou Moussaab",82
1,10612,"ABDEL WADOUD, Abou Mossab",81
2,10612,"ABD EL OUADOUD, Abou Mousab",77
3,10612,"OUDOUD, Abu Musab",76
4,10612,"ABDEL EL-WADOUD, Abu Mossaab",76


In [56]:
from rapidfuzz import process

def search(query, limit=5):
    results = process.extract(query, names["name"],
                              scorer=fuzz.token_sort_ratio,
                              processor=utils.default_process,
                              limit=50)
    rows = []
    for matched_name, score, row_number in results:
        rows.append({"id": names.loc[row_number, "id"],
                     "matched_name": matched_name,
                     "score": round(score)})
    result = pd.DataFrame(rows).drop_duplicates("id")
    return result.head(limit)

search("Musab Abdel Wadoud")

,id,matched_name,score
0,10612,"ABDEL WADOUD, Abou Moussaab",82
7,30884,"AL-SAUDI, Abu Musab",72
18,57858,"ZADA, Sarbaz Abdul",63
19,12403,"SAMAD, Abdul",62
20,13099,"SAMAD, Abdul",62


In [ ]:
Musab Abdel Wadoud